# AE-TFPE Major Revision — Training Campaign

**Run this top to bottom.** It trains models only. It does **not** evaluate on the
test set or on any corruption set — evaluation happens later on your MacBook Pro M4.

## The two method families (never merge them)

| | |
|---|---|
| **Original AE-TFPE** | PE + **ViT-B/16** + **image-space** AE + YOLOv8n-cls. The reference formulation from the submitted manuscript. Reconstructed where the historical implementation was unrecoverable. **Not claimed to be lightweight.** |
| **Efficient AE-TFPE** | PE + **MobileViT-XXS stage 2 (28×28)** + **feature-space** slim denoising AE + YOLOv8n-cls. A **new improvement introduced during the Major Revision** — never presented as the original method. Main candidate: **E5 / C2-28**. |

## Safety properties built into this notebook

- **Google Drive is the source of truth.** Artifacts sync after **every epoch**, so a
  disconnect costs at most one epoch.
- **Resumable.** Re-run the notebook after a disconnect: completed runs are skipped,
  the queue continues.
- **A completed run is never overwritten** without `force=True`.
- **No test access.** `scripts/verify_no_test_access.py` proves at AST level that the
  trainer cannot construct a test or corruption path. Cell 6 runs that proof.

## Two environments — never interchange them

This project runs across two environments with **deliberately different**
dependency stacks. Installing one environment's spec into the other is the
failure mode this section exists to prevent.

### COLAB TRAINING ENVIRONMENT — this notebook

- Uses the **Colab-native, CUDA-compatible `torch` / `torchvision` / `numpy` /
  `Pillow` stack**. These are never downgraded or reinstalled.
- Installs **only training dependencies**, from **`requirements-colab.txt`**
  (via `scripts/colab_setup.sh` in Cell 4).
- **Does not generate corruption datasets.**
- **Does not perform final test evaluation.**

> **Never `pip install -r requirements.txt` on Colab.** That file is the local
> evaluation spec. Its `numpy<2` / `pillow==10.2.0` pins have no wheels for
> current Colab Pythons, so pip source-builds them and the **whole install
> transaction aborts** — leaving `ultralytics` and `timm` missing. The pins are
> also pointless here, because nothing in this notebook touches corruption
> pixels.

### LOCAL EVALUATION ENVIRONMENT — MacBook Pro M4, not this notebook

- Uses the **pinned reproducibility stack** from `requirements.txt`:
  - **Python 3.10.x**
  - **NumPy 1.26.x**
  - **Pillow 10.2.0**
- Used for **corruption generation**, **checksum verification**, and the
  **Normal / Easy / Moderate / Hard evaluation**.
- These pins reproduce `docs/reproducibility_reference.json` exactly. See
  `docs/LOCAL_EVAL_ENVIRONMENT_RECOVERY.md` for how to build that environment.

The split is a **packaging** boundary only. No training or evaluation protocol,
hyperparameter, seed, split, or metric definition differs between them.

## What to do if a cell fails
Every cell below states its failure mode. Nothing is destructive; re-running any cell
is safe.

## CELL 1 — Configuration

The only cell you normally edit. Everything else reads these values.

In [ ]:
# ============================ USER SETTINGS ============================
DRIVE_ROOT   = "/content/drive/MyDrive/AE_TFPE_MajorRevision"   # persistent root
REPO_URL     = "https://github.com/ducthong-dev/VisionTransformer-X-YOLO.git"
REPO_DIR     = "/content/VisionTransformer-X-YOLO"
DATASET_ZIP  = "/content/drive/MyDrive/VisionTransformer_YOLO/dataset/Plant_leaf_diseases_dataset.zip"
DATA_ROOT    = "/content/data/Plant_leaf_diseases_dataset"      # local scratch = fast

# Model-size filter. Models with MORE total parameters than this are SKIPPED by
# default and listed with a reason in Cell 8 -- never silently dropped.
MAX_TRAIN_PARAMS = 20_000_000

# IDs to train anyway despite exceeding the threshold, e.g. ["A5", "D1"].
# Read Cell 8 before setting this: it costs hours per arm.
FORCE_LARGE_IDS = ["A5", "D1", "F1", "F2", "F4"]

# Campaign wall-clock budget. When the remaining budget cannot fit the next run's
# projection, it is marked SKIPPED_TIME rather than started and lost.
CAMPAIGN_BUDGET_HOURS = 20.0

# Hard gate on the five forced fusion arms (A5, D1, F1, F2, F4). Cell 15 trains A5
# first, measures its REAL epoch time on this GPU, projects the whole forced tier,
# and STOPS if the projection exceeds this. It will not quietly consume your day.
FORCED_TIER_MAX_HOURS = 12.0

# DataLoader workers. None = keep the FROZEN value (4). Raising it is a PROTOCOL
# AMENDMENT (see Cell 5) and is deliberately NOT taken: the reproducibility
# protocol stays exactly as frozen.
NUM_WORKERS = None

EPOCHS_OVERRIDE = None    # None = the frozen 30. Use a small number ONLY for a smoke test.
SMOKE_TEST      = False   # True = 4 epochs, 4 images/class. Proves plumbing, not science.
# =======================================================================
import os, time
CAMPAIGN_T0 = time.time()
for k, v in dict(DRIVE_ROOT=DRIVE_ROOT, DATA_ROOT=DATA_ROOT).items():
    print(f"{k:14s} {v}")
print(f"{'MAX_PARAMS':14s} {MAX_TRAIN_PARAMS:,}")
print(f"{'FORCE_LARGE':14s} {FORCE_LARGE_IDS or '(none)'}")
print(f"{'BUDGET':14s} {CAMPAIGN_BUDGET_HOURS} h")
print(f"{'SMOKE_TEST':14s} {SMOKE_TEST}")

## CELL 2 — Mount Google Drive

**Why first:** every artifact is written here. If Drive is not mounted, a disconnect
loses the entire campaign.

**Expected:** `Mounted at /content/drive` and the directory tree printed.

**On failure:** re-run and complete the authorisation popup. Do not proceed without it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ("campaign", "checkpoints", "logs", "configs", "completed", "failed"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("Drive ready:", DRIVE_ROOT)
for sub in sorted(os.listdir(DRIVE_ROOT)):
    p = os.path.join(DRIVE_ROOT, sub)
    n = len(os.listdir(p)) if os.path.isdir(p) else "-"
    print(f"  {sub:<14} {n} entries")

## CELL 3 — Repository

**Why:** every artifact records the commit that produced it. A dirty or unknown
commit makes results untraceable.

**Expected:** a commit SHA and `dirty: False`.

**On failure:** delete `REPO_DIR` and re-run.

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
dirty  = bool(subprocess.check_output(["git", "status", "--porcelain"], text=True).strip())
print("commit:", commit)
print("dirty :", dirty)
print(subprocess.check_output(["git", "log", "-1", "--format=%s"], text=True).strip())
assert not dirty, "Working tree is dirty -- artifacts could not be tied to a commit."

## CELL 4 — Dependencies

**Why:** installs `requirements-colab.txt`, **not** `requirements.txt`.
`requirements.txt` is the *local evaluation* spec — it pins `numpy<2` and
`pillow==10.2.0` so the corruption pixels match
`docs/reproducibility_reference.json`. Current Colab runtimes publish no wheels
for those versions, so pip source-builds them and the **entire** install
transaction aborts — which is how `ultralytics` and `timm` end up missing.

Those pins are also unnecessary here: this notebook **trains only** and cannot
reach corruption or test data (Cell 6 proves it), so no codec determinism is at
stake. Colab's own CUDA-matched `torch` / `numpy` / `pillow` are left untouched.

**Expected:** every package listed with a version, `cuda avail : True`, and
`np<->torch : ok`.

**On failure:** the script prints the offending package and exits non-zero.
Re-run the pip line without `-q` to see the resolver output. If it reports a
broken `np<->torch` bridge, *Runtime → Restart session* and re-run from Cell 2
(Drive stays mounted).


In [ ]:
!bash scripts/colab_setup.sh

# Assert the property the old `numpy == 1.26.*` check was a proxy for: that the
# packages import and the torch<->numpy bridge actually works on this runtime.
import numpy, torch, ultralytics, timm
print("numpy      ", numpy.__version__)
print("torch      ", torch.__version__)
print("ultralytics", ultralytics.__version__)
print("timm       ", timm.__version__)
assert torch.from_numpy(numpy.zeros(4, dtype=numpy.float32)).sum().item() == 0.0, \
    "torch<->numpy bridge is broken -- Runtime > Restart session, then re-run from Cell 2"


## CELL 5 — GPU verification and the optimization audit

**Why:** this campaign targets an **A100**. Training-quality metrics are
hardware-independent, but **training wall-clock is not** — it is recorded and never
presented as architecture evidence.

**The T4 latency/throughput/memory evidence already collected is separate and is not
reproduced here.** A100 numbers must never be substituted for it.

### Optimization audit — every candidate classified

| Optimization | Class | Applied? |
|---|---|---|
| `pin_memory=True` | **SAFE EXECUTION** — page-locked host buffers; no numeric effect. Also makes the existing `non_blocking=True` transfers actually asynchronous | **yes** |
| `persistent_workers=True` | **SAFE EXECUTION** — avoids re-spawning workers each epoch | **yes** |
| `prefetch_factor` | **SAFE EXECUTION** — queue depth only | **yes** |
| Frozen backbone under `no_grad` | **SAFE** — already in the code | already on |
| `num_workers` 4 → 8 | **PROTOCOL AMENDMENT** — each worker owns an RNG stream, so worker count changes *which* augmentation lands on *which* image. Same distribution, fairness preserved when applied uniformly, but **not bit-reproducible** against earlier runs | **NO** — frozen at 4 |
| **AMP / fp16** | **PROTOCOL AMENDMENT** — fp16 changes numerics, and the frozen protocol sets `amp: false` | **NO** |
| `cudnn.benchmark=True` | **PROTOCOL AMENDMENT** — conflicts with `deterministic=True` | **NO** |
| `torch.compile` | **PROTOCOL AMENDMENT** — kernel substitution can change numerics | **NO** |
| **Frozen-feature caching** | **SCIENTIFIC CHANGE — invalid here.** Training augmentation (RandAugment, flip, erasing) is stochastic per epoch, so cached features would not match the augmented input. It would silently train a different model | **NO** |

Only SAFE EXECUTION optimizations are automatic. **No protocol amendment is taken
in this campaign**: `num_workers` stays at the frozen 4, AMP stays off, cuDNN
benchmarking stays off, `torch.compile` is unused, and no feature caching happens.
The reproducibility protocol is exactly as frozen.

### Crash resume — verified

`train.py` writes `last.pt` every epoch containing **model, optimizer, scheduler,
epoch index, best-so-far, the metric history and all RNG states**. On reconnect the
runner copies it back from Drive into the fresh scratch and training continues from
the next epoch. Tested by killing a run mid-training and resuming from Drive alone:
epochs came back contiguous with the best-so-far preserved.

Before this, the only checkpoint held weights alone — no optimizer moments, no LR
schedule position — so a disconnect at epoch 25 of 30 meant restarting from zero.

In [ ]:
import torch, subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                               "--format=csv,noheader"], text=True).strip())
assert torch.cuda.is_available(), "No CUDA device. Runtime -> Change runtime type -> GPU."
GPU = torch.cuda.get_device_name(0)
print("\nGPU        :", GPU)
print("CUDA       :", torch.version.cuda, "| capability", torch.cuda.get_device_capability(0))
print("torch      :", torch.__version__)
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| TF32 cudnn:", torch.backends.cudnn.allow_tf32)
print("cpu cores  :", os.cpu_count())

IS_A100 = "A100" in GPU
if not IS_A100:
    print(f"\nNOTE: this is a {GPU}, not an A100. Training-quality results remain valid;")
    print("      only the wall-clock projections in Cell 9 change.")
print("\nT4 latency/throughput/memory evidence is SEPARATE and is not re-measured here.")

## CELL 6 — Dataset and the no-test-access proof

**Why:** the dataset is unzipped to **local scratch**, not Drive — reading 55k JPEGs
per epoch over Drive would dominate the runtime.

**Expected:** `38,584 / 8,340 / 8,335` across **39 classes**, and the AST proof exits 0.

**On failure:** a different split means the wrong dataset copy (the sibling ResCBAM
copy has 8,346 / 8,334). Fix the path in Cell 1.

In [ ]:
import os
os.environ["DATA_ROOT"]   = DATA_ROOT
os.environ["OUTPUT_ROOT"] = "/content/output"
os.makedirs("/content/data", exist_ok=True); os.makedirs("/content/output", exist_ok=True)

if not os.path.isdir(DATA_ROOT):
    print("unzipping dataset to local scratch ...")
    assert os.path.exists(DATASET_ZIP), f"dataset zip not found: {DATASET_ZIP}"
    import subprocess
    subprocess.run(["unzip", "-q", DATASET_ZIP, "-d", "/content/data"], check=True)
else:
    print("dataset already present")

for split in ("train", "val"):
    d = os.path.join(DATA_ROOT, split)
    n = sum(len(os.listdir(os.path.join(d, c))) for c in os.listdir(d)
            if os.path.isdir(os.path.join(d, c)))
    print(f"  {split:<6} {n:>7,} images, {len(os.listdir(d))} classes")

!python scripts/verify_dataset.py 2>&1 | tail -6
print("\n--- proof that the trainer cannot reach test or corruption data ---")
!python scripts/verify_no_test_access.py

## CELL 7 — Campaign manifest

**Why:** builds the experiment matrix, measures every model's parameters from the real
code, and merges into the Drive manifest. **Finished runs are never downgraded.**

A pre-existing completed **E5** on Drive is detected and adopted — it will **not** be
retrained.

**Expected:** status counts, and any adopted run named.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import importlib, aetfpe.campaign as cp
importlib.reload(cp)

matrix   = cp.build_matrix(max_params=MAX_TRAIN_PARAMS, force_ids=FORCE_LARGE_IDS)
campaign = cp.Campaign(DRIVE_ROOT, scratch_root="/content/campaign_scratch")
campaign.seed(matrix, gpu=GPU)

for rid in [e["id"] for e in cp.EXPERIMENTS]:
    if campaign.adopt_existing(rid):
        r = campaign.manifest["runs"][rid]
        print(f"ADOPTED pre-existing {rid}: best_val_top1={r.get('best_val_top1')} -- not retraining")

print("\nstatus counts:", campaign.counts())
print("P0 queue:", campaign.queue("P0"))
print("P1 queue:", campaign.queue("P1"))
print("P2 queue:", campaign.queue("P2"))
print("\nmanifest:", campaign.manifest_path)

## CELL 8 — Included / skipped models, with reasons

**Nothing is silently skipped.** Read the second table before setting
`FORCE_LARGE_IDS` in Cell 1.

In [ ]:
inc = [r for r in matrix if r["status"] == cp.PENDING]
skp = [r for r in matrix if r["status"] == cp.SKIPPED_SIZE]

print("=" * 118); print("INCLUDED"); print("=" * 118)
print(f"{'ID':<4}{'pri':<5}{'grp':<4}{'model':<34}{'params':>12}{'trainable':>12}  reviewer question")
for r in sorted(inc, key=lambda x: (x["priority"], x["id"])):
    print(f"{r['id']:<4}{r['priority']:<5}{r['group']:<4}{r['title'][:33]:<34}"
          f"{r['params']:>12,}{r['trainable_params']:>12,}  {r['reviewer']}")

print("\n" + "=" * 118); print(f"SKIPPED -- over MAX_TRAIN_PARAMS = {MAX_TRAIN_PARAMS:,}"); print("=" * 118)
print(f"{'ID':<4}{'grp':<4}{'model':<30}{'total':>12}{'trainable':>12}{'frozen':>12}  reviewer question")
for r in sorted(skp, key=lambda x: x["id"]):
    print(f"{r['id']:<4}{r['group']:<4}{r['title'][:29]:<30}{r['params']:>12,}"
          f"{r['trainable_params']:>12,}{r['params']-r['trainable_params']:>12,}  {r['reviewer']}")

print("""
------------------------------------------------------------------------------
READ THIS BEFORE ACCEPTING THE SKIP LIST
------------------------------------------------------------------------------
Seven of the nine skipped arms (A2 A3 A5 D1 F1 F2 F4) carry only ~1.5-1.75M
TRAINABLE parameters. The other ~85.8M is a FROZEN ViT-B/16 that runs under
no_grad -- so the cost is its FORWARD pass, not optimiser work. A total-parameter
threshold therefore over-states how expensive they are to train, while still
correctly flagging them as slow: measured on a T4 they run at 92.8 img/s versus
the baseline's 4,365 img/s (47x).

What the skip costs, per group:
  A2, A3   Original-side component ablation           -> Efficient-side E3/E5 partly covers this
  A5, D1   Original AE-TFPE reference + the denoising
           objective claim (A5 - D1)                  -> NOT covered by anything else
  F1,F2,F4 Fusion comparison (Reviewer #12)           -> NOT covered by anything else
  B1, B3   ResNet-50 / ViT-B/16 external baselines    -> B2 (EfficientNet-B0) covers
           these are FULLY trainable: genuinely expensive  the external-baseline role

Can the skipped arms be represented without training?
  * Complexity / efficiency claims  -> YES. Already measured on a T4 for all five
    architectures and archived in docs/evidence/. Training changes none of it.
  * Original AE-TFPE as a reference -> YES, as a computational/reference formulation
    (see docs/EXPERIMENT_CAMPAIGN_V2_PLAN.md).
  * Fusion superiority (F1/F2/F4 vs D1) -> NO. This is an accuracy comparison and
    cannot be made from architecture analysis. Skipping it means the manuscript
    cannot claim AE fusion is superior; that claim must be withdrawn or deferred.
  * Component ablation on the Original side -> PARTIALLY, via the Efficient side,
    but conclusions are NOT assumed to transfer (different encoder AND AE space).

FLAGGED FOR YOUR DECISION: if you want to keep the AE-fusion superiority claim and
the denoising-objective claim, the minimum additional set is A5 + D1 + F1 + F2 + F4
(five arms). Cell 9 projects what that costs. Set FORCE_LARGE_IDS in Cell 1.
------------------------------------------------------------------------------""")

## CELL 9 — Compute budget projection

**These are estimates until measured.** After the first epoch of each architecture
family the runner prints the *measured* epoch time and re-projects — that is the number
to trust.

Basis: T4 batch-32 throughput (measured), training taken at ~⅓ of inference throughput,
46,924 images per epoch, then adjusted for the detected GPU.

In [ ]:
N_IMG   = 46_924
EPOCHS  = EPOCHS_OVERRIDE or 30
T4_IPS  = {"small": 4365.6, "mobilevit": 774.8, "vit": 92.8}   # measured, batch 32
SPEEDUP = 3.0 if IS_A100 else 1.0                              # rough A100:T4 for fp32
LOADER_FLOOR_S = 75.0     # see note below

def family(r):
    if r["params"] > 20_000_000:                        return "vit"
    if "mobilevit" in r["config"] or r["group"] == "E": return "mobilevit"
    return "small"

def train_multiplier(r):
    """Training cost / inference cost.

    Backward runs only over TRAINABLE parameters. The ViT-B/16 arms keep 96.7% of
    their forward FLOPs in a frozen branch executed under no_grad, so a flat
    'training = 3x inference' rule overstates them by ~2.9x. Measured split for
    A5: 33.7266 frozen + 1.1465 trainable of 34.8731 GFLOPs -> 1.066x, not 3x.
    """
    share = r["trainable_params"] / max(r["params"], 1)
    return 1.0 + 2.0 * share

def project(r):
    ips = T4_IPS[family(r)] / train_multiplier(r) * SPEEDUP
    compute_s = N_IMG / ips
    # Small models are usually data-loader bound, not compute bound: 46,924 JPEG
    # decodes + RandAugment per epoch on 4 workers. Take whichever dominates.
    return max(compute_s, LOADER_FLOOR_S) * EPOCHS / 3600.0

print(f"GPU {GPU} | epochs {EPOCHS} | {N_IMG:,} images/epoch | assumed A100:T4 speedup {SPEEDUP}x")
print(f"data-loader floor assumed {LOADER_FLOOR_S:.0f} s/epoch at num_workers=4\n")
tot = {}
for pri in ("P0", "P1", "P2"):
    rows = [r for r in matrix if r["priority"] == pri and r["status"] == cp.PENDING]
    h = sum(project(r) for r in rows)
    tot[pri] = h
    print(f"{pri}: {len(rows)} runs -> ~{h:.1f} h   " + ", ".join(r["id"] for r in rows))
print(f"\nP0+P1 projected: ~{tot['P0']+tot['P1']:.1f} h   (budget {CAMPAIGN_BUDGET_HOURS} h)")

big = [r for r in matrix if r["status"] == cp.SKIPPED_SIZE]
print(f"\nIf you forced the skipped arms instead:")
for r in sorted(big, key=lambda x: x["id"]):
    print(f"   {r['id']:<4}{r['title'][:34]:<36} ~{project(r):>5.1f} h")
print(f"   {'ALL':<4}{'(not recommended in one day)':<36} ~{sum(project(r) for r in big):>5.1f} h")
print(f"   {'A5+D1+F1+F2+F4':<40} ~{sum(project(r) for r in big if r['id'] in ('A5','D1','F1','F2','F4')):>5.1f} h")
print("""
ESTIMATES ONLY -- the numbers to trust are the MEASURED epoch times the runner
prints after each first epoch. Two things drive the estimates above:
  * backward runs only over TRAINABLE parameters, so the ViT-B/16 arms cost about
    1.07x their inference cost, not 3x (96.7% of their forward is frozen);
  * small models are data-loader bound, so P0 runs take a similar wall-clock
    regardless of parameter count.
The forced tier is additionally gated on its own measured epoch time in Cell 15.""")

## CELL 10 — Training runner

**Why:** one function so every arm gets identical treatment. It prints the frozen
protocol, applies only SAFE execution optimizations, syncs to Drive after every epoch,
and re-projects the remaining campaign from the *measured* first-epoch time.

**On failure:** the run is marked `FAILED` with its log path; the campaign continues to
the next arm. Re-running the cell retries failed runs.

In [ ]:
import csv, json, time

def protocol_banner():
    from aetfpe.config import load_experiment, build_protocol
    c = load_experiment("configs/baseline_rgb.yaml"); p = build_protocol(c)
    tr = os.path.join(DATA_ROOT, "train"); va = os.path.join(DATA_ROOT, "val")
    ntr = sum(len(os.listdir(os.path.join(tr,x))) for x in os.listdir(tr) if os.path.isdir(os.path.join(tr,x)))
    nva = sum(len(os.listdir(os.path.join(va,x))) for x in os.listdir(va) if os.path.isdir(os.path.join(va,x)))
    print("=" * 78); print("FROZEN PROTOCOL -- identical for every comparable arm"); print("=" * 78)
    for k, v in [("dataset", DATA_ROOT), ("train images", f"{ntr:,}"), ("val images", f"{nva:,}"),
                 ("classes", len(os.listdir(tr))), ("image size", p.img_size),
                 ("epochs", EPOCHS_OVERRIDE or p.epochs), ("batch size", p.batch_size),
                 ("optimizer", p.optimizer), ("lr", p.lr), ("weight decay", p.weight_decay),
                 ("scheduler", "cosine + 3-epoch warm-up"), ("seed", f"{p.seed} (deterministic={p.deterministic})"),
                 ("augmentation", "hflip 0.5, RandAugment(2,9), RandomErasing 0.4"),
                 ("checkpoint", p.checkpoint_selection), ("AMP", f"{p.amp}  (frozen off)"),
                 ("pretrained", "True (ImageNet / COCO transfer)"),
                 ("AE warm-up", f"{p.ae_warmup_epochs} reconstruction-only epochs (AE arms)"),
                 ("num_workers", f"{NUM_WORKERS or p.num_workers}"),
                 ("TEST EVALUATION", "NONE -- validation only, done later on the Mac")]:
        print(f"  {k:<16} {v}")
    print("=" * 78)

def train_one(rid, force=False):
    extra = []
    if NUM_WORKERS:  extra += ["--num-workers", str(NUM_WORKERS)]
    if SMOKE_TEST:   extra += ["--limit-per-class", "4"]
    epochs = 4 if SMOKE_TEST else EPOCHS_OVERRIDE
    used_h = (time.time() - CAMPAIGN_T0) / 3600.0
    left_h = CAMPAIGN_BUDGET_HOURS - used_h
    row = [r for r in matrix if r["id"] == rid]
    proj = project(row[0]) if row else 0.0
    if not SMOKE_TEST and proj > left_h:
        campaign.manifest["runs"][rid].update(
            status=cp.SKIPPED_TIME,
            reason=f"projected {proj:.1f} h exceeds the {left_h:.1f} h remaining")
        campaign.save()
        print(f"[{rid}] SKIPPED_TIME -- projected {proj:.1f} h > {left_h:.1f} h remaining")
        return campaign.manifest["runs"][rid]

    t0 = time.time()
    r = campaign.run(rid, epochs=epochs, extra_args=extra, force=force, gpu=GPU)

    m = os.path.join(campaign.ckpt_dir, rid, "metrics.csv")
    if os.path.exists(m):
        rows = list(csv.DictReader(open(m)))
        if rows:
            e1 = float(rows[0]["seconds"])
            n  = EPOCHS_OVERRIDE or 30
            print(f"[{rid}] MEASURED epoch 1: {e1:.1f}s -> projected {e1*n/3600:.2f} h "
                  f"for {n} epochs (estimate was {proj:.2f} h)")
    print(f"[{rid}] campaign elapsed {(time.time()-CAMPAIGN_T0)/3600:.2f} h "
          f"of {CAMPAIGN_BUDGET_HOURS} h\n")
    return r

def run_tier(pri):
    q = campaign.queue(pri)
    print(f"\n########## {pri}: {len(q)} run(s) -> {q}\n")
    for rid in q:
        train_one(rid)
    print(f"########## {pri} done. counts: {campaign.counts()}")

def show_summary():
    import pandas as pd
    df = pd.read_csv(campaign.summary_path)
    cols = [c for c in ["id","model","priority","status","params","best_val_top1",
                        "best_val_top5","runtime_s","gpu"] if c in df.columns]
    display(df[cols])
    done = df[df.status == "COMPLETED"]
    print(f"COMPLETED {len(done)} | elapsed {(time.time()-CAMPAIGN_T0)/3600:.2f} h")
    p0 = df[(df.priority == "P0")]
    ok = len(p0[p0.status == "COMPLETED"])
    print(f"P0: {ok}/{len(p0)} complete " +
          ("<<< SCIENTIFIC MINIMUM COMPLETE >>>" if ok == len(p0) and len(p0) else "(incomplete)"))

protocol_banner()
print("\nrunner ready: train_one(id) | run_tier('P0') | show_summary()")

## CELL 11 — P0: the scientific minimum

**These six determine whether the revised paper is defensible.**

`A0` fair baseline · `E5` Efficient AE-TFPE · `M1/M2/M3` mechanism controls · `E3` AE-removed control.

**The mechanism gate:** if E5 does not clearly beat the best of M1/M2/M3, the
contribution is the input transform, not the architecture — and that is the finding.
Stop and report it rather than spending the rest of the budget.

In [ ]:
run_tier('P0')

## CELL 12 — Checkpoint after P0

In [ ]:
show_summary()

## CELL 13 — P1: high value

`A1` PE-only (serves **both** method families — with no TF branch the Original and
Efficient variants are the same model) · `A4` RGB+AE · `E7` C2-7 spatial control ·
`B2` EfficientNet-B0 external baseline.

**Do not skip to Cell 15 before this finishes.** The forced P2 arms are the
expensive ones; P1 is cheap and answers component questions.

In [ ]:
run_tier('P1')

## CELL 14 — Checkpoint after P1

In [ ]:
show_summary()

## CELL 15 — P2: optional, only if time remains

**Forced this campaign: `A5, D1, F1, F2, F4`** — the five arms that carry the
AE-fusion superiority claim and the denoising-objective claim (A5 − D1). Without
them those claims must be withdrawn or deferred.

`B1` (ResNet-50) and `B3` (ViT-B/16) stay skipped: both are **fully trainable** and
therefore genuinely expensive, and `B2` (EfficientNet-B0) already fills the external
-baseline role.

Each forced arm carries the frozen ViT-B/16, so `last.pt` is ~365 MB and is written
to Drive every epoch. Budget roughly **11 GB of Drive writes per arm** and ~3.5 GB of
resident Drive space across the five.

Runs whose projection does not fit the remaining budget are marked `SKIPPED_TIME`
rather than started and lost to a timeout.

In [ ]:
q = campaign.queue("P2")
if not q:
    print("P2 queue empty -- nothing forced. Set FORCE_LARGE_IDS in Cell 1 to add arms.")
else:
    # GATE: train ONE forced arm first, measure it, then decide with real numbers
    # instead of an estimate. A5 is the representative -- D1/F1/F2/F4 share its
    # frozen ViT-B/16 branch and are within a few percent of its cost.
    probe = "A5" if "A5" in q else q[0]
    print(f"GATE: training {probe} first to measure the real cost of this tier.\n")
    train_one(probe)

    import csv
    mp = os.path.join(campaign.ckpt_dir, probe, "metrics.csv")
    rows = list(csv.DictReader(open(mp))) if os.path.exists(mp) else []
    if not rows:
        print(f"No metrics for {probe}; not proceeding blindly. Inspect logs/{probe}.log.")
    else:
        ep_s = sum(float(r["seconds"]) for r in rows) / len(rows)
        per_run_h = ep_s * (EPOCHS_OVERRIDE or 30) / 3600.0
        remaining = [r for r in campaign.queue("P2")]
        tier_h = per_run_h * len(remaining)
        print(f"\nMEASURED {probe}: {ep_s:.0f} s/epoch -> {per_run_h:.2f} h per forced arm")
        print(f"{len(remaining)} arm(s) left ({remaining}) -> {tier_h:.2f} h")
        print(f"gate: FORCED_TIER_MAX_HOURS = {FORCED_TIER_MAX_HOURS} h")

        if tier_h > FORCED_TIER_MAX_HOURS:
            print(f"""
STOP -- the forced tier projects {tier_h:.1f} h, over the {FORCED_TIER_MAX_HOURS} h gate.
Not started. {probe} is COMPLETED and safe on Drive.

Your options, in order of scientific value:
  1. Run D1 only  (~{per_run_h:.1f} h). With A5 already done, A5-D1 isolates the
     DENOISING OBJECTIVE -- the specific claim Reviewer #10.4 challenges.
     This is the highest-value single arm remaining.
  2. Add F1 and F2 (~{2*per_run_h:.1f} h more) for a partial fusion comparison,
     disclosed as partial.
  3. Raise FORCED_TIER_MAX_HOURS and re-run this cell if you have the wall-clock.

To run a specific arm:  train_one("D1")""")
        else:
            print(f"\nGATE PASSED -- {tier_h:.1f} h fits the {FORCED_TIER_MAX_HOURS} h budget. Continuing.\n")
            run_tier("P2")

## CELL 16 — Final campaign summary

In [ ]:
show_summary()
import pandas as pd
df = pd.read_csv(campaign.summary_path)
print("\nby status:"); print(df.status.value_counts().to_string())
print(f"\nmanifest : {campaign.manifest_path}")
print(f"summary  : {campaign.summary_path}")
print(f"checkpts : {campaign.ckpt_dir}")
print("\nNo test evaluation was run. No corruption set was generated.")

## CELL 17 — Export for local evaluation on the MacBook Pro M4

Copies only what evaluation needs. Full details in `docs/LOCAL_EVALUATION_HANDOFF.md`.

**Each completed run's folder contains:** `checkpoint.pt` (best val top-1, with its
config and class list embedded), `config.yaml`, `metrics.csv`, `train_summary.json`,
`environment.json`.

In [ ]:
import tarfile, os, json
out = os.path.join(DRIVE_ROOT, "campaign", "for_local_evaluation.tar.gz")
done = [rid for rid, r in campaign.manifest["runs"].items() if r.get("status") == "COMPLETED"]
with tarfile.open(out, "w:gz") as tar:
    for rid in sorted(done):
        d = os.path.join(campaign.ckpt_dir, rid)
        if os.path.isdir(d):
            tar.add(d, arcname=rid)
    tar.add(campaign.summary_path, arcname="campaign_summary.csv")
    tar.add(campaign.manifest_path, arcname="campaign_manifest.json")
print(f"wrote {out}  ({os.path.getsize(out)/1e6:.1f} MB)  runs: {sorted(done)}")
print(f"""
NEXT, ON YOUR MAC:
  1. Download {out} from Drive
  2. mkdir -p results/campaign && tar -xzf for_local_evaluation.tar.gz -C results/campaign
  3. Follow docs/LOCAL_EVALUATION_HANDOFF.md for Normal / Easy / Moderate / Hard

Evaluation was deliberately NOT run here.""")